In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df=pd.read_excel('Zomato Chennai Listing 2020.xlsx')


In [ ]:
#basic explorations
print(df.shape)
print(df.info())
print(df.dtypes)
print(df.isnull().sum())
print(df.duplicated().sum())

(12032, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12032 entries, 0 to 12031
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Zomato URL             12032 non-null  object 
 1   Name of Restaurant     12032 non-null  object 
 2   Address                12032 non-null  object 
 3   location               12032 non-null  object 
 4   Cuisine                12032 non-null  object 
 5   Top Dishes             12032 non-null  object 
 6   Price for 2            12032 non-null  int64  
 7   Dining Rating          6681 non-null   float64
 8   Dining Rating Count    11812 non-null  object 
 9   Delivery Rating        6181 non-null   float64
 10  Delivery Rating Count  11812 non-null  object 
 11  Features               12032 non-null  object 
dtypes: float64(2), int64(1), object(9)
memory usage: 1.1+ MB
None
Zomato URL                object
Name of Restaurant        object
Address           

In [ ]:
#removing useless columns
df=df.drop(columns=['Address','Zomato URL'])
print(df.columns)

Index(['Name of Restaurant', 'location', 'Cuisine', 'Top Dishes',
       'Price for 2', 'Dining Rating', 'Dining Rating Count',
       'Delivery Rating', 'Delivery Rating Count', 'Features'],
      dtype='object')


In [ ]:
duplicates=df[df.duplicated(subset=['Name of Restaurant'],keep=False)]
print(duplicates[['Name of Restaurant','location']].sort_values(by=['Name of Restaurant','location']).head(50))

                         Name of Restaurant                         location
6081                             180 Degree                        Sowcarpet
7177                             180 Degree                           Vepery
3279                             7th Heaven                  Anna Nagar West
7346                             7th Heaven  Gokulam Park Hotel, Ashok Nagar
3098                             7th Heaven                         RA Puram
2130                             7th Heaven                        Velachery
6731                               90s Kids                         GST Road
4717                               90s Kids                      Kodambakkam
11236  A.M.S Hyderabad Biriyani & Fast Food                      Kodambakkam
11235  A.M.S Hyderabad Biriyani & Fast Food                        Koyambedu
11234  A.M.S Hyderabad Biriyani & Fast Food                         T. Nagar
2260   A.M.S Hyderabad Biriyani & Fast Food                       Vadapalani

In [ ]:
df.rename(columns={'Name of Restaurant':'Name','Address':'address','Cuisine':'cuisine','Top Dishes':'top dishes','Price for 2':'price for 2','Dining Rating':'dining rating','Dining Rating Count':'dining rating count','Delivery Rating':'delivery rating','Delivery Rating Count':'delivery rating count'},inplace=True)

In [ ]:
df = df.drop_duplicates(subset=['Name', 'location'], keep='first')
print(df.shape)


(11561, 10)


In [ ]:
df['delivery rating'].unique()

array([4.3, 4.1, 4.4, 4. , 3.8, 4.2, 3.9, 3.6, nan, 3.3, 3.4, 3.7, 3.5,
       4.5, 2.9, 4.6, 3.1, 2.7, 3.2, 3. , 2.8, 2.3, 2.4, 4.7, 2.6, 2.5,
       2.2, 2.1, 2. , 1.8, 1.6, 1.5, 0.3])

In [ ]:
null_percent = df['delivery rating'].isnull().sum() / len(df) * 100
print(null_percent)

48.283020499956756


In [ ]:
df['dining rating'].unique()

array([4.3, 4.4, 4. , 4.2, 4.1, nan, 3.9, 4.5, 4.7, 3.5, 4.6, 3.6, 3.8,
       4.9, 4.8, 3.7, 2.7, 2.9, 2.6, 3. , 3.4, 2.4, 3.3, 2.8, 2.1, 2. ,
       2.2, 3.2, 2.5, 3.1, 2.3, 1.9, 1.7, 0.3, 1. ])

In [ ]:
null_percent = df['dining rating'].isnull().sum() / len(df) * 100
print(null_percent)

44.0273332756682


In [ ]:
# Reset has_dine_in based on dining rating directly
df['has_dine_in'] = df['dining rating'].notna()

# Reset has_delivery based on delivery rating directly
df['has_delivery'] = df['delivery rating'].notna()

# Verify
print(df['has_dine_in'].value_counts())
print(df['has_delivery'].value_counts())

has_dine_in
True     6471
False    5090
Name: count, dtype: int64
has_delivery
True     5979
False    5582
Name: count, dtype: int64


In [ ]:
print(f"No dine-in restaurants: {df['has_dine_in'].value_counts()[False]}")
print(f"Remaining NaN dining rating: {df['dining rating'].isnull().sum()}")

print(f"No delivery restaurants: {df['has_delivery'].value_counts()[False]}")
print(f"Remaining NaN delivery rating: {df['delivery rating'].isnull().sum()}")

No dine-in restaurants: 5090
Remaining NaN dining rating: 5090
No delivery restaurants: 5582
Remaining NaN delivery rating: 5582


In [ ]:
zero_engagement = df[df['dining rating count'].isnull()]
print(zero_engagement[['Name', 'location', 'dining rating', 'delivery rating', 'dining rating count', 'delivery rating count']].head(10))

                                          Name        location  dining rating  \
1603                          Erode Amman Mess   Thiruvanmiyur            NaN   
2159  The Ultimate Brownie And Chocolate Place  Sholinganallur            NaN   
2419                    Amudha Aunty's kitchen  Sholinganallur            NaN   
2537                                TN 03 CAFE    Tiruvottiyur            NaN   
2542                               Maggi Point        Tambaram            NaN   
2559                           Hotspot Kitchen        Ambattur            NaN   
2612                  Mr. Lee Fungs By Novotel  Sholinganallur            NaN   
2755                               Maggi Point         Potheri            NaN   
3032                                  Bowlsome        T. Nagar            NaN   
3191                                 Snack Box        RA Puram            NaN   

      delivery rating dining rating count delivery rating count  
1603              NaN                 NaN 

In [ ]:
df = df.dropna(subset=['dining rating count', 'delivery rating count'], how='all')
print(df.shape)

(11345, 12)


In [ ]:
# Confirm 216 rows dropped
print(df.isnull().sum())

Name                        0
location                    0
cuisine                     0
top dishes                  0
price for 2                 0
dining rating            4874
dining rating count         0
delivery rating          5366
delivery rating count       0
Features                    0
has_dine_in                 0
has_delivery                0
dtype: int64


In [ ]:
min_price_for_2=min(df['price for 2'])
min_price_for_2

40

In [ ]:
max_price_for_2=df['price for 2'].max()
max_price_for_2

5000

In [ ]:
mean_price_for_2 = df['price for 2'].mean()
mean_price_for_2

np.float64(401.4596738651388)

In [ ]:
df['price for 2'].describe()

,price for 2
count,11345.000000
mean,401.459674
std,339.112136
min,40.000000
25%,200.000000
50%,300.000000
75%,450.000000
max,5000.000000


In [ ]:
Q1=df['price for 2'].quantile(0.25)
Q3=df['price for 2'].quantile(0.75)
IQR=Q3-Q1
IQR

np.float64(250.0)

In [ ]:
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(lower_bound)
print(upper_bound)

-175.0
825.0


In [ ]:
(df['price for 2'] > upper_bound).sum()

np.int64(623)

In [ ]:
df['price for 2'].clip(upper=upper_bound)

,price for 2
0,500
1,825
2,500
3,500
4,450
...,...
12011,500
12012,500
12013,500
12014,500


In [ ]:
# How many restaurants are above 825?
print(df[df['price for 2'] > 825].shape[0])

# What do they look like?
print(df[df['price for 2'] > 825][['Name','location','price for 2']].head(20))

623
                               Name                        location  \
1                Sukkubhai Biriyani                         Alandur   
8                 Savoury Sea Shell                 Anna Nagar East   
13           Dindigul Thalappakatti                           Porur   
17                           Abid's                         Chetpet   
18          Wire Room Bar & Kitchen  Phoenix Market City, Velachery   
23                          BFF 2.0                        RA Puram   
30                          Zaitoon                      Royapettah   
42                       Orange Wok                    Nungambakkam   
45                        Pedreno's                        Mylapore   
47                        Palmshore                          Egmore   
49                      Sea Emperor                      Madipakkam   
51                 Paradise Biryani                   Thiruvanmiyur   
53                   Copper Kitchen                      Saligramam   
67

In [ ]:
# Just convert to numeric safely — no outlier removal needed
df['price for 2'] = pd.to_numeric(df['price for 2'], errors='coerce')

# Verify
print(df['price for 2'].describe())
print(df['price for 2'].isnull().sum())

count    11345.000000
mean       401.459674
std        339.112136
min         40.000000
25%        200.000000
50%        300.000000
75%        450.000000
max       5000.000000
Name: price for 2, dtype: float64
0


In [ ]:
result=[]
for i in df['price for 2']:
  if i>=1 and i<=300:
    result.append('Budget')
  elif i>=301 and i<=600:
    result.append('Affordable')
  elif i>=601 and i<=1000:
    result.append('Mid Range')
  else:
    result.append('Premium')
df['price category']=result
df

,Name,location,cuisine,top dishes,price for 2,dining rating,dining rating count,delivery rating,delivery rating count,Features,has_dine_in,has_delivery,price category
0,Yaa Mohaideen Briyani,Pallavaram,['Biryani'],"['Bread Halwa', ' Chicken 65', ' Mutton Biryan...",500,4.3,1500,4.3,9306,"['Home Delivery', 'Indoor Seating']",True,True,Affordable
1,Sukkubhai Biriyani,Alandur,"['Biryani', ' North Indian', ' Mughlai', ' Des...","['Beef Biryani', ' Beef Fry', ' Paratha', ' Pa...",1000,4.4,3059,4.1,39200,"['Home Delivery', 'Free Parking', 'Table booki...",True,True,Mid Range
2,SS Hyderabad Biryani,Kodambakkam,"['Biryani', ' North Indian', ' Chinese', ' Ara...","['Brinjal Curry', ' Tandoori Chicken', ' Chick...",500,4.3,1361,4.4,10500,"['Home Delivery', 'Indoor Seating']",True,True,Affordable
3,KFC,Perambur,"['Burger', ' Fast Food', ' Finger Food', ' Bev...",['Zinger Burger'],500,4.0,1101,4.0,11200,"['Home Delivery', 'Free Parking', 'Card Upon D...",True,True,Affordable
4,Tasty Kitchen,Perambur,"['Chinese', ' Biryani', ' North Indian', ' Che...","['Mutton Biryani', ' Chicken Rice', ' Tomato R...",450,4.2,617,4.1,22400,"['Home Delivery', 'Indoor Seating']",True,True,Affordable
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12011,Bowl Bazaar,Anna Nagar East,"['North Indian', ' South Indian', ' Chinese', ...",Invalid,500,NaN,Does not offer Dining,NaN,Not enough Delivery Reviews,['Delivery Only'],False,False,Affordable
12012,Bowl Bazaar,Ashok Nagar,"['North Indian', ' South Indian', ' Chinese', ...",Invalid,500,NaN,Does not offer Dining,NaN,Not enough Delivery Reviews,['Delivery Only'],False,False,Affordable
12013,Bowl Bazaar,Perungudi,"['North Indian', ' South Indian', ' Chinese', ...",Invalid,500,NaN,Does not offer Dining,NaN,Not enough Delivery Reviews,['Delivery Only'],False,False,Affordable
12014,Bowl Bazaar,Adyar,"['North Indian', ' South Indian', ' Chinese', ...",Invalid,500,NaN,Does not offer Dining,NaN,Not enough Delivery Reviews,['Delivery Only'],False,False,Affordable


In [ ]:
df['location']=df['location'].str.strip().str.title()
df['location'].unique()


array(['Pallavaram', 'Alandur', 'Kodambakkam', 'Perambur', 'Medavakkam',
       'Navallur', 'Anna Nagar East', 'T. Nagar', 'Velachery',
       'Vadapalani', 'Porur', 'Kilpauk', 'Purasavakkam', 'Ashok Nagar',
       'Chetpet', 'Phoenix Market City, Velachery', 'Aminijikarai',
       'Ramapuram', 'Mylapore', 'Thuraipakkam', 'Ra Puram', 'Alwarpet',
       'Mogappair', 'West Mambalam', 'Adyar', 'Nungambakkam',
       'Royapettah', 'Chromepet', 'Kolathur', 'Valasaravakkam', 'Guindy',
       'Potheri', 'Egmore', 'Madipakkam', 'Thiruvanmiyur', 'Saligramam',
       'Sholinganallur', 'Perungudi', 'Besant Nagar', 'Kotturpuram',
       'Semmancheri', 'Royapuram', 'Anna Nagar West',
       'Grand By Grt Hotels', 'Ambattur', 'Thousand Lights',
       'Mayajaal Multiplex, Kanathur', 'Washermenpet', 'Selaiyur',
       'Old Mahabalipuram Road (Omr)', 'The Westin Chennai, Velachery',
       'The Park, Nungambakkam', 'Choolaimedu',
       'New Woodlands Hotel, Mylapore', 'Abhiramapuram', 'Triplicane',
 

In [ ]:
df['cuisine'].astype(str).str.strip()

,cuisine
0,['Biryani']
1,"['Biryani', ' North Indian', ' Mughlai', ' Des..."
2,"['Biryani', ' North Indian', ' Chinese', ' Ara..."
3,"['Burger', ' Fast Food', ' Finger Food', ' Bev..."
4,"['Chinese', ' Biryani', ' North Indian', ' Che..."
...,...
12011,"['North Indian', ' South Indian', ' Chinese', ..."
12012,"['North Indian', ' South Indian', ' Chinese', ..."
12013,"['North Indian', ' South Indian', ' Chinese', ..."
12014,"['North Indian', ' South Indian', ' Chinese', ..."


In [ ]:
df['cuisine_list']=df['cuisine'].apply(eval) # to proper conversion of list

In [ ]:
exploded_df=df.explode('cuisine_list')
exploded_df['has_dine_in'] = exploded_df['has_dine_in'].astype(int)
exploded_df['has_delivery'] = exploded_df['has_delivery'].astype(int)
exploded_df

,Name,location,cuisine,top dishes,price for 2,dining rating,dining rating count,delivery rating,delivery rating count,Features,has_dine_in,has_delivery,price category,cuisine_list
0,Yaa Mohaideen Briyani,Pallavaram,['Biryani'],"['Bread Halwa', ' Chicken 65', ' Mutton Biryan...",500,4.3,1500,4.3,9306,"['Home Delivery', 'Indoor Seating']",1,1,Affordable,Biryani
1,Sukkubhai Biriyani,Alandur,"['Biryani', ' North Indian', ' Mughlai', ' Des...","['Beef Biryani', ' Beef Fry', ' Paratha', ' Pa...",1000,4.4,3059,4.1,39200,"['Home Delivery', 'Free Parking', 'Table booki...",1,1,Mid Range,Biryani
1,Sukkubhai Biriyani,Alandur,"['Biryani', ' North Indian', ' Mughlai', ' Des...","['Beef Biryani', ' Beef Fry', ' Paratha', ' Pa...",1000,4.4,3059,4.1,39200,"['Home Delivery', 'Free Parking', 'Table booki...",1,1,Mid Range,North Indian
1,Sukkubhai Biriyani,Alandur,"['Biryani', ' North Indian', ' Mughlai', ' Des...","['Beef Biryani', ' Beef Fry', ' Paratha', ' Pa...",1000,4.4,3059,4.1,39200,"['Home Delivery', 'Free Parking', 'Table booki...",1,1,Mid Range,Mughlai
1,Sukkubhai Biriyani,Alandur,"['Biryani', ' North Indian', ' Mughlai', ' Des...","['Beef Biryani', ' Beef Fry', ' Paratha', ' Pa...",1000,4.4,3059,4.1,39200,"['Home Delivery', 'Free Parking', 'Table booki...",1,1,Mid Range,Desserts
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12014,Bowl Bazaar,Adyar,"['North Indian', ' South Indian', ' Chinese', ...",Invalid,500,NaN,Does not offer Dining,NaN,Not enough Delivery Reviews,['Delivery Only'],0,0,Affordable,Biryani
12015,Bowl Bazaar,Medavakkam,"['North Indian', ' South Indian', ' Chinese', ...",Invalid,500,NaN,Does not offer Dining,NaN,Not enough Delivery Reviews,['Delivery Only'],0,0,Affordable,North Indian
12015,Bowl Bazaar,Medavakkam,"['North Indian', ' South Indian', ' Chinese', ...",Invalid,500,NaN,Does not offer Dining,NaN,Not enough Delivery Reviews,['Delivery Only'],0,0,Affordable,South Indian
12015,Bowl Bazaar,Medavakkam,"['North Indian', ' South Indian', ' Chinese', ...",Invalid,500,NaN,Does not offer Dining,NaN,Not enough Delivery Reviews,['Delivery Only'],0,0,Affordable,Chinese


In [ ]:

exploded_df = df.explode('cuisine_list').copy()

exploded_df['cuisine_list'] = exploded_df['cuisine_list'].str.strip()

# Convert boolean columns to int
exploded_df['has_dine_in'] = exploded_df['has_dine_in'].astype(int)
exploded_df['has_delivery'] = exploded_df['has_delivery'].astype(int)

# Verify
print(exploded_df['cuisine_list'].value_counts().head(20))
print(exploded_df.shape)

# Check conversion
print(exploded_df[['has_dine_in', 'has_delivery']].head())

cuisine_list
Chinese         3430
North Indian    3163
Fast Food       2968
South Indian    2912
Beverages       2337
Biryani         1807
Desserts        1249
Bakery           856
Street Food      836
Ice Cream        742
Chettinad        628
Sandwich         627
Italian          438
Pizza            410
Juices           381
Burger           349
Continental      329
Seafood          320
Arabian          314
Mithai           308
Name: count, dtype: int64
(27471, 14)
   has_dine_in  has_delivery
0            1             1
1            1             1
1            1             1
1            1             1
1            1             1


In [ ]:
top_20_common_dishes=exploded_df.value_counts().head(20)
top_20_common_dishes

Name         location     cuisine                                                          top dishes                                                                                        price for 2  dining rating  dining rating count  delivery rating  delivery rating count  Features                                                                      has_dine_in  has_delivery  price category  cuisine_list
nEATo Foods  Ambattur     ['South Indian', ' North Indian', ' Street Food', ' Fast Food']  Invalid                                                                                           300          3.3            6                    3.8              120                    ['Breakfast', 'Home Delivery', 'Vegetarian Only', 'Outdoor Seating']          1            1             Budget          Street Food     1
101 Dosa     Velachery    ['Fast Food', ' South Indian']                                   Invalid                                                                                           300          3.6            64                   3.8              261                    ['Home Delivery', 'Vegetarian Only', 'Free Parking', 'Indoor Seating']        1            1             Budget          Fast Food       1
                                                                                                                                                                                                                                                                                                                                                                                                               South Indian    1
11 O Cafe    Perungudi    ['Cafe']                                                         ['Sandwich', ' Pizza', ' Burgers', ' Hot Chocolate', ' Chilli Fries', ' Chicken Burger', ' Tea']  650          3.8            135                  3.6              67                     ['Home Delivery', 'Indoor Seating']                                           1            1             Mid Range       Cafe            1
ibaco        Perambur     ['Ice Cream']                                                    Invalid                                                                                           200          3.8            73                   4.3              562                    ['Home Delivery', 'Vegetarian Only', 'Indoor Seating', 'Desserts and Bakes']  1            1             Budget          Ice Cream       1
             Pallavaram   ['Ice Cream']                                                    Invalid                                                                                           200          3.9            96                   4.4              346                    ['Home Delivery', 'Indoor Seating', 'Desserts and Bakes']                     1            1             Budget          Ice Cream       1
             Okkiyampet   ['Ice Cream']                                                    ['Icecream Cake', ' Belgian Chocolate', ' Cream Cake']                                            200          4.2            250                  4.4              1485                   ['Home Delivery', 'Vegetarian Only', 'Indoor Seating', 'Desserts and Bakes']  1            1             Budget          Ice Cream       1
             Neelangarai  ['Ice Cream']                                                    Invalid                                                                                           200          3.9            57                   4.2              195                    ['Home Delivery', 'Vegetarian Only', 'Indoor Seating', 'Desserts and Bakes']  1            1             Budget          Ice Cream       1
             Navallur     ['Ice Cream']                                                    Invalid                                                                                           200          3.9            77                   4.4           

In [ ]:
# Just strip — no title case for restaurant names
df['Name'] = df['Name'].str.strip()

# Verify
print(df['Name'].head(10))

0       Yaa Mohaideen Briyani
1          Sukkubhai Biriyani
2        SS Hyderabad Biryani
3                         KFC
4               Tasty Kitchen
5                  Dine N Fun
6          Bai Veetu Kalyanam
7                Cafe Arabica
8           Savoury Sea Shell
9    Sangeetha Veg Restaurant
Name: Name, dtype: object


In [ ]:
df.shape

(11345, 14)

In [ ]:
df.isnull().sum()

,0
Name,1
location,0
cuisine,0
top dishes,0
price for 2,0
dining rating,4874
dining rating count,0
delivery rating,5366
delivery rating count,0
Features,0


In [ ]:
df.head()

,Name,location,cuisine,top dishes,price for 2,dining rating,dining rating count,delivery rating,delivery rating count,Features,has_dine_in,has_delivery,price category,cuisine_list
0,Yaa Mohaideen Briyani,Pallavaram,['Biryani'],"['Bread Halwa', ' Chicken 65', ' Mutton Biryan...",500,4.3,1500,4.3,9306,"['Home Delivery', 'Indoor Seating']",True,True,Affordable,[Biryani]
1,Sukkubhai Biriyani,Alandur,"['Biryani', ' North Indian', ' Mughlai', ' Des...","['Beef Biryani', ' Beef Fry', ' Paratha', ' Pa...",1000,4.4,3059,4.1,39200,"['Home Delivery', 'Free Parking', 'Table booki...",True,True,Mid Range,"[Biryani, North Indian, Mughlai, Desserts, ..."
2,SS Hyderabad Biryani,Kodambakkam,"['Biryani', ' North Indian', ' Chinese', ' Ara...","['Brinjal Curry', ' Tandoori Chicken', ' Chick...",500,4.3,1361,4.4,10500,"['Home Delivery', 'Indoor Seating']",True,True,Affordable,"[Biryani, North Indian, Chinese, Arabian]"
3,KFC,Perambur,"['Burger', ' Fast Food', ' Finger Food', ' Bev...",['Zinger Burger'],500,4.0,1101,4.0,11200,"['Home Delivery', 'Free Parking', 'Card Upon D...",True,True,Affordable,"[Burger, Fast Food, Finger Food, Beverages]"
4,Tasty Kitchen,Perambur,"['Chinese', ' Biryani', ' North Indian', ' Che...","['Mutton Biryani', ' Chicken Rice', ' Tomato R...",450,4.2,617,4.1,22400,"['Home Delivery', 'Indoor Seating']",True,True,Affordable,"[Chinese, Biryani, North Indian, Chettinad,..."


In [ ]:
df['has_dine_in'] = df['has_dine_in'].astype(int)
df['has_delivery'] = df['has_delivery'].astype(int)

# Verify
print(df['has_dine_in'].value_counts())
print(df['has_delivery'].value_counts())

has_dine_in
1    6471
0    4874
Name: count, dtype: int64
has_delivery
1    5979
0    5366
Name: count, dtype: int64


In [ ]:
# Fix column names - replace spaces with underscores
df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower()

# Fix dining_rating_count - remove text values
df['dining_rating_count'] = pd.to_numeric(df['dining_rating_count'], errors='coerce').fillna(0).astype(int)

# Fix NaN in rating columns
df['dining_rating'] = df['dining_rating'].fillna(0)
df['delivery_rating'] = df['delivery_rating'].fillna(0)

# Verify
print(df.columns.tolist())
print(df.dtypes)

['name', 'location', 'cuisine', 'top_dishes', 'price_for_2', 'dining_rating', 'dining_rating_count', 'delivery_rating', 'delivery_rating_count', 'features', 'has_dine_in', 'has_delivery', 'price_category', 'cuisine_list']
name                      object
location                  object
cuisine                   object
top_dishes                object
price_for_2                int64
dining_rating            float64
dining_rating_count        int64
delivery_rating          float64
delivery_rating_count     object
features                  object
has_dine_in                int64
has_delivery               int64
price_category            object
cuisine_list              object
dtype: object


In [ ]:
# Reorder columns to match table
df = df[['name', 'location', 'cuisine', 'top_dishes', 'price_for_2',
         'dining_rating', 'dining_rating_count', 'delivery_rating',
         'delivery_rating_count', 'features', 'price_category',
         'cuisine_list', 'has_dine_in', 'has_delivery']]

print(df.columns.tolist())

['name', 'location', 'cuisine', 'top_dishes', 'price_for_2', 'dining_rating', 'dining_rating_count', 'delivery_rating', 'delivery_rating_count', 'features', 'price_category', 'cuisine_list', 'has_dine_in', 'has_delivery']


In [ ]:
df['delivery_rating_count'] = pd.to_numeric(df['delivery_rating_count'], errors='coerce').fillna(0).astype(int)

In [ ]:
print("Column order:", df.columns.tolist())
print("\nShape:", df.shape)
print("\nPrice category counts:", df['price_category'].value_counts())
print("\nDining rating nulls:", df['dining_rating'].isnull().sum())
print("\nDelivery rating nulls:", df['delivery_rating'].isnull().sum())
print("\nDining rating count nulls:", df['dining_rating_count'].isnull().sum())
print("\nDtypes:\n", df.dtypes)

Column order: ['name', 'location', 'cuisine', 'top_dishes', 'price_for_2', 'dining_rating', 'dining_rating_count', 'delivery_rating', 'delivery_rating_count', 'features', 'price_category', 'cuisine_list', 'has_dine_in', 'has_delivery']

Shape: (11345, 14)

Price category counts: price_category
Budget        5981
Affordable    4186
Mid Range      801
Premium        377
Name: count, dtype: int64

Dining rating nulls: 0

Delivery rating nulls: 0

Dining rating count nulls: 0

Dtypes:
 name                      object
location                  object
cuisine                   object
top_dishes                object
price_for_2                int64
dining_rating            float64
dining_rating_count        int64
delivery_rating          float64
delivery_rating_count      int64
features                  object
price_category            object
cuisine_list              object
has_dine_in                int64
has_delivery               int64
dtype: object


In [ ]:
df.to_csv('zomato_chennai_cleaned.csv', index=False)
exploded_df.to_csv('zomato_cuisine_exploded.csv', index=False)
print(f"Phase 2 Complete! ")
print(f"Main cleaned file: {df.shape}")
print(f"Exploded cuisine file: {exploded_df.shape}")

Phase 2 Complete! 
Main cleaned file: (11345, 14)
Exploded cuisine file: (27471, 14)
